# 🎬 Clipper: turn long videos into shorts
Turns a podcast, stream or interview into captioned vertical shorts (9:16) for TikTok, Reels and YouTube Shorts. Everything runs on Google's free cloud computers, so you don't need anything but a browser.

**How to use (works on a phone):**
1. Turn on a GPU (optional, but much faster): **Runtime → Change runtime type → T4 GPU**.
2. Tap ▶ on each step below, from top to bottom. Wait for each step to finish before starting the next.
3. Download your clips at the end.

For smarter clip picking, add an Anthropic API key: tap the 🔑 **Secrets** icon in the left sidebar, add `ANTHROPIC_API_KEY`, and switch on notebook access. Without a key, the tool still works with a simpler built-in picker.

In [ ]:
#@title 1️⃣ Set up (about 1–2 minutes)
BRANCH = "ccr-f8f1b6dc-utsp0q"  #@param {type:"string"}
import os, subprocess
os.chdir("/content")
if not os.path.exists("/content/Claude-ai"):
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                        "https://github.com/freedomdanceuk-ai/Claude-ai.git"])
    if r.returncode != 0:  # branch merged/deleted: use the default branch
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/freedomdanceuk-ai/Claude-ai.git"], check=True)
else:
    subprocess.run(["git", "-C", "/content/Claude-ai", "pull", "-q"])
!pip install -q -r /content/Claude-ai/clipper/requirements.txt yt-dlp
os.chdir("/content/Claude-ai")

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ Claude API key loaded: AI clip picking is on")
except Exception:
    print("ℹ️ No ANTHROPIC_API_KEY secret found: the simpler built-in picker will be used")
import torch
print("✅ GPU available" if torch.cuda.is_available() else "⚠️ No GPU: still works, just slower (Runtime → Change runtime type → T4 GPU)")
print("✅ Setup done")

In [ ]:
#@title 2️⃣ Choose your video
#@markdown **upload**: pick a file from your phone or computer (best for files under about 500 MB).<br>
#@markdown **drive**: a file in your Google Drive (best for big files). Enter its path, e.g. `MyDrive/podcast.mp4`.<br>
#@markdown **url**: a link to a video you own or have permission to clip.
SOURCE = "upload"  #@param ["upload", "drive", "url"]
DRIVE_PATH = "MyDrive/podcast.mp4"  #@param {type:"string"}
VIDEO_URL = ""  #@param {type:"string"}

import glob
os.makedirs("/content/input", exist_ok=True)
if SOURCE == "upload":
    from google.colab import files
    os.chdir("/content/input")  # files.upload() saves into the current folder
    try:
        VIDEO = f"/content/input/{next(iter(files.upload()))}"
    finally:
        os.chdir("/content/Claude-ai")
elif SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = f"/content/drive/{DRIVE_PATH}"
else:
    !yt-dlp -q -f "bv*[height<=1080]+ba/b[height<=1080]" --merge-output-format mp4 -o "/content/input/%(id)s.%(ext)s" "$VIDEO_URL"
    VIDEO = max(glob.glob("/content/input/*.mp4"), key=os.path.getmtime)

assert os.path.exists(VIDEO), f"Video not found: {VIDEO}"
print(f"✅ Using {VIDEO} ({os.path.getsize(VIDEO) / 1e6:.0f} MB)")

In [ ]:
#@title 3️⃣ Make the clips
NUMBER_OF_CLIPS = 5  #@param {type:"slider", min:1, max:20, step:1}
MIN_SECONDS = 20  #@param {type:"integer"}
MAX_SECONDS = 60  #@param {type:"integer"}
#@markdown **crop**: fill the screen and follow the speaker's face. **blur**: show the full frame over a blurred background (good for 2+ people).
REFRAME = "crop"  #@param ["crop", "blur", "none"]
#@markdown Larger models transcribe more accurately but take longer. Use `medium` or `large-v3` with a GPU, and `small` without one.
WHISPER_MODEL = "medium"  #@param ["small", "medium", "large-v3"]
CAPTIONS = True  #@param {type:"boolean"}

import shutil
os.makedirs("/content/clips", exist_ok=True)
for old in glob.glob("/content/clips/*.mp4") + glob.glob("/content/clips/*.ass"):
    os.remove(old)  # keep the cached transcript so re-runs with new settings are fast
args = ["python", "-m", "clipper", VIDEO, "-o", "/content/clips", "-n", str(NUMBER_OF_CLIPS),
        "--min", str(MIN_SECONDS), "--max", str(MAX_SECONDS),
        "--reframe", REFRAME, "--whisper-model", WHISPER_MODEL]
if not CAPTIONS:
    args.append("--no-captions")
!{" ".join(f'"{a}"' for a in args)}

In [ ]:
#@title 4️⃣ Preview the clips
import json, base64
from IPython.display import HTML, display
manifest = json.load(open("/content/clips/clips.json"))
for i, c in enumerate(manifest, 1):
    data = base64.b64encode(open(f"/content/clips/{c['file']}", "rb").read()).decode()
    display(HTML(f'''<div style="margin:12px 0"><b>#{i} {c['title']}</b> · score {c['score']} · {c['end'] - c['start']:.0f}s<br>
<i>{c['reason']}</i><br><video controls playsinline width="270" src="data:video/mp4;base64,{data}"></video></div>'''))

In [ ]:
#@title 5️⃣ Download all clips (zip)
from google.colab import files
shutil.make_archive("/content/clips", "zip", "/content/clips")
files.download("/content/clips.zip")
#@markdown On a phone, the zip lands in your Downloads folder. To save clips to Google Drive instead, run the next step.

In [ ]:
#@title 6️⃣ (Optional) Save clips to Google Drive
from google.colab import drive
drive.mount("/content/drive")
dest = "/content/drive/MyDrive/Clipper"
os.makedirs(dest, exist_ok=True)
for f in glob.glob("/content/clips/*.mp4") + ["/content/clips/clips.json"]:
    shutil.copy(f, dest)
print(f"✅ Saved to Google Drive → MyDrive/Clipper")